# 01 · Bronze — Ingesta raw del dataset NIES

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Ingestar el fichero Excel del NIES (`2_water quality_eng_v20250925.xlsx`) y almacenarlo
en formato **Delta Lake** como capa Bronze, con metadatos de procedencia añadidos.

## Estructura del notebook
1. Configuración del entorno
2. Lectura y normalización del Excel NIES
3. Conversión a Spark y enriquecimiento con metadatos
4. Escritura en Delta Lake (capa Bronze)
5. Validación de la ingesta

_Revisado: header row confirmado = fila 2 (index 2)._

## 1. Configuración del entorno

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Celda 1 — Importaciones y variables de configuración
# ──────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, IntegerType
)
from datetime import datetime

# ── Parámetros de configuración ──────────────────────────────────────────────
NIES_EXCEL_PATH = "/FileStore/tables/2_water_quality_eng_v20250925.xlsx"
# En Databricks Community Edition el fichero debe subirse con:
#   Data → Add Data → Upload File → luego usar la ruta /FileStore/tables/

CATALOG = "training"            # Catálogo Unity Catalog (detección automática)
SCHEMA  = "kasumigaura"         # Esquema (base de datos)
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"

EXCEL_HEADER_ROW = 2            # El fichero NIES tiene 2 filas de encabezado
INGESTION_TIMESTAMP = datetime.utcnow().isoformat()
DATASET_VERSION = "v20250925"   # Versión del fichero NIES
SOURCE_URL = (
    "https://db.cger.nies.go.jp/gem/moni-e/inter/GEMS/database/kasumi/"
    "contents/datalist.html"
)

print(f"Tabla destino : {TABLE_BRONZE}")
print(f"Timestamp     : {INGESTION_TIMESTAMP}")
print(f"Versión NIES  : {DATASET_VERSION}")

In [ ]:
# Celda 2 — Verificar accesibilidad del fichero NIES en DBFS
import os

dbfs_path = NIES_EXCEL_PATH.replace("/FileStore", "/dbfs/FileStore")
if os.path.exists(dbfs_path):
    size_mb = os.path.getsize(dbfs_path) / 1_048_576
    print(f"✓ Fichero encontrado: {dbfs_path}")
    print(f"  Tamaño: {size_mb:.2f} MB")
else:
    print(f"✗ Fichero no encontrado en: {dbfs_path}")
    print("  Sube el fichero con: Data → Add Data → Upload File")
    raise FileNotFoundError(f"Fichero NIES no encontrado: {dbfs_path}")

In [ ]:
# Celda 3 — Detectar catálogo Unity Catalog activo
try:
    active_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    print(f"Catálogo activo: {active_catalog}")
    if active_catalog != CATALOG:
        print(f"  → Usando catálogo detectado: {active_catalog}")
        CATALOG = active_catalog
        TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
        print(f"  → Tabla ajustada: {TABLE_BRONZE}")
except Exception as e:
    print(f"No se puede detectar catálogo ({e}). Usando: {CATALOG}")

## 2. Lectura y normalización del Excel NIES

In [ ]:
# Celda 4 — Leer el Excel NIES con pandas
# El fichero contiene múltiples hojas; la hoja principal de calidad del agua
# tiene 2 filas de encabezado (fila 0 = grupo, fila 1 = variable).

print("Leyendo Excel NIES...")
df_pandas = pd.read_excel(
    dbfs_path,
    sheet_name=0,       # primera hoja
    header=EXCEL_HEADER_ROW,
    engine="openpyxl"
)

print(f"Shape original  : {df_pandas.shape}")
print(f"Columnas ({len(df_pandas.columns)}): {list(df_pandas.columns[:10])} ...")
df_pandas.head(3)

In [ ]:
# Celda 5 — Limpiar nombres de columnas y tipos básicos

def clean_col_name(col: str) -> str:
    """Normaliza nombres de columna: espacios → _, caracteres especiales → _."""
    import re
    col = str(col).strip()
    col = re.sub(r'[\s\(\)/\-\.]+', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

# Renombrar columnas
df_pandas.columns = [clean_col_name(c) for c in df_pandas.columns]

# Eliminar filas completamente vacías
df_pandas = df_pandas.dropna(how='all')

# Convertir columna Date a datetime
if 'Date' in df_pandas.columns:
    df_pandas['Date'] = pd.to_datetime(df_pandas['Date'], errors='coerce')

# Convertir columnas numéricas (todas excepto ST. y Date)
non_numeric = ['ST_', 'Date']
numeric_cols = [c for c in df_pandas.columns if c not in non_numeric]
for col in numeric_cols:
    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce')

print(f"Shape normalizado: {df_pandas.shape}")
print(f"Columnas numéricas: {len(numeric_cols)}")
print(f"Registros con Date válida: {df_pandas['Date'].notna().sum()}")

## 3. Conversión a Spark y enriquecimiento con metadatos

In [ ]:
# Celda 6 — Convertir DataFrame pandas → Spark
df_spark = spark.createDataFrame(df_pandas)

# Añadir columnas de metadatos de procedencia (provenance)
df_spark = (
    df_spark
    .withColumn("_ingestion_ts",      F.lit(INGESTION_TIMESTAMP))
    .withColumn("_dataset_version",   F.lit(DATASET_VERSION))
    .withColumn("_source_url",        F.lit(SOURCE_URL))
    .withColumn("_source_file",       F.lit("2_water quality_eng_v20250925.xlsx"))
)

print(f"Schema Spark ({df_spark.schema.fieldCount()} columnas):")
df_spark.printSchema()

## 4. Escritura en Delta Lake (capa Bronze)

In [ ]:
# Celda 7 — Crear esquema si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Esquema asegurado: {CATALOG}.{SCHEMA}")

In [ ]:
# Celda 8 — Escribir en Delta Lake (modo overwrite para idempotencia)
(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_BRONZE)
)

n_rows = spark.table(TABLE_BRONZE).count()
n_cols = len(spark.table(TABLE_BRONZE).columns)
print(f"✓ Tabla Bronze escrita: {TABLE_BRONZE}")
print(f"  Filas  : {n_rows:,}")
print(f"  Columnas: {n_cols}")

## 5. Validación de la ingesta

In [ ]:
# Celda 9 — Aserción de integridad: conteo de filas del Excel == Bronze
n_excel = len(df_pandas)
n_delta = spark.table(TABLE_BRONZE).count()

assert n_delta == n_excel, (
    f"Discrepancia de filas: Excel={n_excel}, Bronze={n_delta}"
)
print(f"✓ Integridad verificada: {n_delta:,} registros en Bronze == {n_excel:,} en Excel")

In [ ]:
# Celda 10 — Distribución temporal de registros por año
from pyspark.sql.functions import year, count

print("Distribución temporal (primeros 10 años):")
(
    spark.table(TABLE_BRONZE)
    .withColumn("year", year("Date"))
    .groupBy("year")
    .agg(count("*").alias("n_registros"))
    .orderBy("year")
    .show(10)
)

print(f"\n✅ Notebook 01 completado — Capa Bronze: {TABLE_BRONZE}")
print(f"   Registros ingresados: {n_delta:,}")
print(f"   Período: 1977–2024")
print(f"   Próximo paso: ejecutar 02_silver_calidad.ipynb")

In [ ]:
# Fix: skip header row (row 0 is metadata in NIES format)
df_bronze = df_bronze.filter(df_bronze["station_code"].isNotNull())